# 03 Baseline model
Week 4: a Linear Regression baseline that every later model has to beat by a margin that isn't noise (handbook §09).
CRISP-DM Modeling phase: test design written before training, then build, describe and assess the model.

Input: the cleaned splits from `02_preprocessing.ipynb` (`dataset/processed/`).

#### Setup

In [1]:
import json
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

SEED = 42  # LinearRegression is deterministic; fixed anyway for anything random added later (handbook §11)
np.random.seed(SEED)

DATA_DIR = os.path.join("dataset", "processed")
MODEL_DIR = os.path.join(DATA_DIR, "models")  # inside dataset/, which .gitignore excludes
os.makedirs(MODEL_DIR, exist_ok=True)
TARGET = "ClosePrice"

#### Test design (written before training)
- **Split** (handbook §01): train = Jan 2024 – Feb 2026, validation = Mar 2026, test = Apr 2026. Chronological, no shuffling.
- **Choices** (which features, which variant) are made on **validation** only.
- **Test is not used here.** Per handbook §01 it's touched once, in Week 8, where the baseline and every later model are scored on it together. Loading it here would make it tempting to check after each model.
- **Metrics** (handbook §08): MdAPE is the headline; R², MAPE and MAE reported alongside.
- **Error breakdown**: by price band (quintiles of the actual price) and by county.
- **Success criterion**: the baseline has no target to hit; it sets the bar for Weeks 5–7. Later models must beat its validation MdAPE by a clear margin.

#### Step 1: Load the cleaned splits

In [2]:
def load_split(split):
    df = pd.read_csv(os.path.join(DATA_DIR, f"{split}_clean.csv"), dtype={"PostalCode": str}, low_memory=False)
    df["CloseDate"] = pd.to_datetime(df["CloseDate"])
    # some ZIPs are ZIP+4 ('92651-1234'); keep the 5-digit ZIP so the same area is one category
    df["PostalCode"] = df["PostalCode"].str[:5]
    return df


data = {s: load_split(s) for s in ["train", "val"]}  # test is held out until Week 8
{s: d.shape for s, d in data.items()}

{'train': (280488, 38), 'val': (11343, 38)}

In [3]:
# Feature list comes from the dataset description made in 02, so roles are defined in one place.
# Excluded: id, target, comps_only (price_per_sqft contains the target; CloseDate, property_id are for comps).
description = pd.read_csv(os.path.join(DATA_DIR, "dataset_description.csv"))
features = description.loc[description["role"].isin(["model_feature", "missing_flag"]), "column"].tolist()

CAT_COLS = ["Levels", "CountyOrParish", "City", "PostalCode"]
# right-skewed sizes: log1p before scaling (see Step 4)
LOG_COLS = ["LivingArea", "LotSizeSquareFeet", "GarageSpaces", "AssociationFee"]
NUM_COLS = [c for c in features if c not in CAT_COLS + LOG_COLS]
print(f"{len(LOG_COLS)} logged: {LOG_COLS}")
print(f"{len(NUM_COLS)} numeric: {NUM_COLS}")
print(f"{len(CAT_COLS)} categorical: {CAT_COLS}")

4 logged: ['LivingArea', 'LotSizeSquareFeet', 'GarageSpaces', 'AssociationFee']
23 numeric: ['BedroomsTotal', 'BathroomsTotalInteger', 'YearBuilt', 'HasHOA', 'Latitude', 'Longitude', 'ViewYN', 'PoolPrivateYN', 'FireplaceYN', 'NewConstructionYN', 'AgeAtSale', 'month_sin', 'month_cos', 'LotSizeSquareFeet_missing', 'GarageSpaces_missing', 'YearBuilt_missing', 'BedroomsTotal_missing', 'BathroomsTotalInteger_missing', 'AssociationFee_missing', 'ViewYN_missing', 'PoolPrivateYN_missing', 'FireplaceYN_missing', 'NewConstructionYN_missing']
4 categorical: ['Levels', 'CountyOrParish', 'City', 'PostalCode']


#### Step 2: Metrics

In [4]:
def metrics(y_true, y_pred):
    ape = np.abs(y_pred - y_true) / y_true  # absolute percentage error per home
    return {
        "MdAPE_%": round(100 * np.median(ape), 2),  # headline: error on the typical home
        "MAPE_%": round(100 * np.mean(ape), 2),
        "R2": round(r2_score(y_true, y_pred), 4),
        "MAE_$": round(mean_absolute_error(y_true, y_pred)),
        "within_10%": round(100 * np.mean(ape <= 0.10), 1),  # share of homes predicted within 10%
    }


def error_by(df, y_pred, groups):
    # metrics per group (e.g. price band or county)
    out = df.assign(pred=y_pred).groupby(groups, observed=True)
    rows = {g: {"n": len(d), **metrics(d[TARGET].to_numpy(), d["pred"].to_numpy())} for g, d in out}
    return pd.DataFrame(rows).T

#### Step 3: Model 0, median price
A sanity floor: predict the train median price for every home. Any real model has to beat this easily.

In [5]:
train_median = data["train"][TARGET].median()
results = {"median_price": metrics(data["val"][TARGET].to_numpy(), np.full(len(data["val"]), train_median))}
results["median_price"]

{'MdAPE_%': np.float64(39.46),
 'MAPE_%': np.float64(49.37),
 'R2': -0.1034,
 'MAE_$': 575984,
 'within_10%': np.float64(13.8)}

#### Step 4: Model 1, Linear Regression
- **Pipeline** (handbook §07): scaling and one-hot encoding live inside a `ColumnTransformer`, so `fit` only ever sees train.
- **Log target**: prices are right-skewed and errors scale with price, so the model predicts `log(ClosePrice)` and the prediction is converted back with `exp`. This fits percentage error, which is what MdAPE measures.
- **Log the skewed sizes** (LivingArea, LotSizeSquareFeet, GarageSpaces, AssociationFee) with `log1p`. With a log target and raw sizes, each extra sq ft multiplies the price, so a 15,000 sqft home got a $100M+ prediction. Logged sizes make the effect "x% more area -> y% more price", which stays sensible at the extremes and also tames bad values like a 1.9 billion sqft lot.
- **Categoricals**: one-hot, with categories seen fewer than 30 times in train grouped as "infrequent". Unseen categories in val/test go to that group instead of erroring.
- **Two variants** to see how much location detail helps a linear model:
  - `lr_county`: property features + lat/long + county
  - `lr_zip`: the same + ZIP code (City is left out; ZIP is the finer version of the same information)

Target encoding for City/ZIP (handbook §05) waits for Week 6.

In [6]:
def build_lr(cat_cols):
    pre = ColumnTransformer([
        ("log", Pipeline([("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                          ("scale", StandardScaler())]), LOG_COLS),
        ("num", StandardScaler(), NUM_COLS),
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30), cat_cols),
    ])
    pipe = Pipeline([("pre", pre), ("lr", LinearRegression())])
    return TransformedTargetRegressor(regressor=pipe, func=np.log, inverse_func=np.exp)


variants = {
    "lr_county": ["Levels", "CountyOrParish"],
    "lr_zip": ["Levels", "CountyOrParish", "PostalCode"],
}
X = {s: d[features] for s, d in data.items()}
y = {s: d[TARGET].to_numpy() for s, d in data.items()}

models, val_pred = {}, {}
for name, cat_cols in variants.items():
    models[name] = build_lr(cat_cols).fit(X["train"], y["train"])  # TRAIN ONLY
    val_pred[name] = models[name].predict(X["val"])
    results[name] = metrics(y["val"], val_pred[name])

# train scores too: a big train/val gap would mean overfitting
for name in variants:
    results[f"{name} (train)"] = metrics(y["train"], models[name].predict(X["train"]))

val_results = pd.DataFrame(results).T
val_results

,MdAPE_%,MAPE_%,R2,MAE_$,within_10%
median_price,39.46,49.37,-0.1034,575984.0,13.8
lr_county,17.68,22.53,0.6541,291819.0,30.6
lr_zip,9.47,13.37,0.8346,176082.0,52.1
lr_county (train),17.42,22.49,0.6766,287123.0,30.4
lr_zip (train),9.29,12.88,0.8683,167079.0,53.0


In [7]:
# pick the variant with the lowest validation MdAPE
chosen = min(variants, key=lambda n: results[n]["MdAPE_%"])
print("chosen on validation:", chosen)

chosen on validation: lr_zip


#### Step 5: Error breakdown on validation (handbook §08)

In [8]:
val = data["val"]
pred = val_pred[chosen]

# price band: quintiles of the actual price
bands = pd.qcut(val[TARGET], 5, labels=["Q1 lowest", "Q2", "Q3", "Q4", "Q5 highest"])
error_by(val.assign(price_band=bands), pred, "price_band")

,n,MdAPE_%,MAPE_%,R2,MAE_$,within_10%
Q1 lowest,2294.0,9.31,14.86,-0.1258,62098.0,53.1
Q2,2254.0,7.95,11.65,-2.6736,79170.0,58.4
Q3,2258.0,8.52,11.64,-2.7540,104415.0,57.1
Q4,2280.0,9.50,12.42,-1.1433,162680.0,51.8
Q5 highest,2257.0,12.82,16.28,0.5145,473954.0,40.3


In [9]:
# by county, largest 10 counties in validation
top_counties = val["CountyOrParish"].value_counts().index[:10]
in_top = val["CountyOrParish"].isin(top_counties)
error_by(val[in_top], pred[in_top.to_numpy()], "CountyOrParish").sort_values("n", ascending=False)

,n,MdAPE_%,MAPE_%,R2,MAE_$,within_10%
Los Angeles,2778.0,10.09,13.77,0.7950,202163.0,49.7
Riverside,1913.0,9.20,14.50,0.6123,143179.0,53.2
San Diego,1268.0,9.36,12.63,0.8708,186228.0,53.1
San Bernardino,1185.0,7.97,11.45,0.7912,67662.0,58.8
Orange,1095.0,10.09,12.75,0.7969,237985.0,49.9
Contra Costa,538.0,8.35,12.44,0.8090,171095.0,55.4
Alameda,491.0,11.17,15.30,0.7343,227471.0,46.0
Ventura,368.0,8.73,13.87,0.7296,195116.0,54.9
Santa Clara,322.0,9.06,11.71,0.8697,274125.0,52.5
San Luis Obispo,202.0,11.70,14.40,0.7250,176653.0,44.6


In [10]:
# Model description: largest coefficients (on log price, scaled numeric inputs).
# A coefficient of 0.10 means roughly +10% price for +1 standard deviation (numeric) or for that category.
lr = models[chosen].regressor_
names = lr.named_steps["pre"].get_feature_names_out()
coefs = pd.Series(lr.named_steps["lr"].coef_, index=names)
numeric_coefs = coefs[coefs.index.str.startswith(("num__", "log__"))].sort_values()
numeric_coefs

num__Longitude                       -0.376304
num__Latitude                        -0.369855
num__BedroomsTotal                   -0.011229
num__month_cos                       -0.009301
num__HasHOA                          -0.008644
num__FireplaceYN_missing             -0.002020
num__BathroomsTotalInteger_missing   -0.001119
num__NewConstructionYN_missing       -0.000131
num__BedroomsTotal_missing            0.000000
num__YearBuilt_missing                0.000321
num__LotSizeSquareFeet_missing        0.000609
num__GarageSpaces_missing             0.001581
num__ViewYN_missing                   0.002663
log__AssociationFee                   0.002745
num__month_sin                        0.003272
num__FireplaceYN                      0.003794
num__AssociationFee_missing           0.010202
num__NewConstructionYN                0.012714
num__ViewYN                           0.014526
log__GarageSpaces                     0.019180
num__PoolPrivateYN_missing            0.020301
num__PoolPriv

#### Step 6: Save the model and results

In [11]:
# Everything the Week 9 app needs to rebuild a prediction: the fitted pipeline and the exact input columns
joblib.dump(models[chosen], os.path.join(MODEL_DIR, "baseline_lr.joblib"))
with open(os.path.join(MODEL_DIR, "baseline_lr_features.json"), "w") as f:
    json.dump({"model": chosen, "features": features, "categorical": variants[chosen], "logged": LOG_COLS,
               "seed": SEED}, f, indent=2)

val_results.to_csv(os.path.join(MODEL_DIR, "baseline_metrics.csv"))  # validation only
print("saved to", MODEL_DIR)

saved to dataset/processed/models


#### Findings
- **Baseline to beat (validation)**: `lr_zip`, MdAPE **9.46%**, MAPE 13.37%, R² 0.835, MAE $176k; 52% of homes within 10%. Test is scored once in Week 8, with all models together.
- **Location does most of the work for a linear model**: adding ZIP cut MdAPE from 17.7% (`lr_county`) to 9.5%. Lat/long on their own are linear, and price doesn't change linearly across California, so ZIP one-hot does what lat/long can't.
- **No overfitting**: train and validation scores are within about 0.2 MdAPE points.
- **Weakest on the most expensive homes**: the top price quintile has MdAPE 12.8%, compared with 8–9.5% for the others. R² inside one price band is often negative because each band covers a narrow price range, so use MdAPE/MAPE to compare bands, not R².
- **Logging the sizes was necessary**: with raw LivingArea and lot size, a few 14–17k sqft homes were predicted at $50–127M, and validation R² was −0.99 even though MdAPE looked fine. That's why all four metrics get reported.
- **Coefficients**: log LivingArea dominates (+21% per standard deviation), then YearBuilt/AgeAtSale and bathrooms. YearBuilt and AgeAtSale are almost the same variable (AgeAtSale = sale year − YearBuilt), so their separate coefficients shouldn't be read on their own.
- **Data issues seen while debugging** (not fixed here; belong in 02): a train home with 45 bedrooms and 45 bathrooms, GarageSpaces up to 600, and lot sizes up to 1.9 billion sqft all pass the current step 2 rules.
- **Next (Weeks 5–7)**: tree models (no need to log inputs, handle nonlinear location), target encoding for City/ZIP, then comps features.